# Ingest — GitHub Copilot AI credits, via API

The only one of the four sources that can be **fully automated**. Viva, Studio and Entra all
require someone to download a file; this one does not.

Uses the dedicated AI-credit endpoints rather than the general billing usage endpoint:

```
GET /enterprises/{enterprise}/copilot/billing/seats            -> licensed users
GET /enterprises/{enterprise}/settings/billing/ai_credit/usage -> usage, filtered per user
```

**The structural catch.** `user` is a *request filter*, not a field on each returned row. The API
tells you what a given user consumed but does not name them in the response. So we enumerate seats
first and loop, tagging each row with the login we asked about. At 500 developers that is 500
calls, against a 5,000/hour limit.

**Two things the API cannot give you**, both present in the emailed CSV: `total_monthly_quota` and
the `aic_*` columns. Neither is in the API schema, and neither is documented in the CSV reference
either. CreditLens does not depend on them — the pooled allowance is computed from the seat list
instead — but if you need them, keep taking the CSV.

**Classic PAT only.** Fine-grained tokens are not supported on the billing endpoints. Scope
`read:enterprise` or `manage_billing:copilot`, held by an enterprise admin or billing manager.

In [ ]:
ENTERPRISE = "your-enterprise-slug"

# Put the classic PAT in a Key Vault and read it through a linked service.
# Never paste it into the notebook - notebooks are committed, secrets should not be.
#   TOKEN = notebookutils.credentials.getSecret("https://<vault>.vault.azure.net/", "github-billing-pat")
TOKEN = notebookutils.credentials.getSecret("https://YOUR-VAULT.vault.azure.net/", "github-billing-pat")

# Months to pull. Backfill up to 24; then run monthly with BACKFILL_MONTHS = 2
# so the current month and the one before it are both refreshed - late-posting
# usage would otherwise be missed.
BACKFILL_MONTHS = 2

TBL_USAGE = "github_ai_usage"
TBL_SEATS = "github_user_map"

API = "https://api.github.com"
HEADERS = {
    "Accept": "application/vnd.github+json",
    "Authorization": f"Bearer {TOKEN}",
    "X-GitHub-Api-Version": "2022-11-28",
}

In [ ]:
import requests
import time
from datetime import date
from dateutil.relativedelta import relativedelta
from pyspark.sql import functions as F, Row
from delta.tables import DeltaTable


def get(url, params=None, retries=4):
    """GET with backoff. Honours GitHub's own retry hints rather than guessing:
    403/429 with x-ratelimit-remaining: 0 means wait until the reset time."""
    for attempt in range(retries):
        r = requests.get(url, headers=HEADERS, params=params, timeout=60)
        if r.status_code == 200:
            return r
        if r.status_code in (403, 429):
            if r.headers.get("x-ratelimit-remaining") == "0":
                wait = max(int(r.headers.get("x-ratelimit-reset", 0)) - int(time.time()), 1)
                print(f"  rate limited, sleeping {wait}s")
                time.sleep(min(wait, 900))
                continue
            time.sleep(int(r.headers.get("retry-after", 2 ** attempt)))
            continue
        if r.status_code >= 500:
            time.sleep(2 ** attempt)
            continue
        r.raise_for_status()
    raise RuntimeError(f"{url} failed after {retries} attempts: {r.status_code} {r.text[:300]}")

## Seats

Also gives us the seat list CreditLens needs for cost — plan and included credits per developer.

In [ ]:
seats, page = [], 1
while True:
    r = get(f"{API}/enterprises/{ENTERPRISE}/copilot/billing/seats",
            {"per_page": 100, "page": page})
    batch = r.json().get("seats", [])
    if not batch:
        break
    seats.extend(batch)
    page += 1

print(f"{len(seats):,} Copilot seats")

# plan_type comes back as business / enterprise; CreditLens keys the seat-price
# measure on the display name, so normalise it here rather than in DAX.
PLAN = {"business": "Copilot Business", "enterprise": "Copilot Enterprise"}

seat_rows = [
    Row(username=s["assignee"]["login"],
        user_principal_name=None,   # see the note below
        display_name=s["assignee"].get("name") or s["assignee"]["login"],
        plan=PLAN.get((s.get("plan_type") or "").lower(), "Copilot Business"),
        included_credits=None)      # filled from Settings, see below
    for s in seats
]
seat_df = spark.createDataFrame(seat_rows)
seat_df.groupBy("plan").count().show()

### Two columns the API cannot fill

**`user_principal_name`** is what joins a GitHub account to the Entra org file, and so to every
department breakdown in the report. GitHub does not know it. Options, best first:

1. **SAML/SCIM identity mapping** — if your enterprise uses SAML SSO, query
   `GET /enterprises/{enterprise}/consumed-licenses`, which returns the SAML `name_id` per user.
   That is usually the UPN, and this is the only fully automatic route.
2. **A mapping table** you maintain in the Lakehouse — reliable, needs upkeep.
3. **Guess from the email pattern** — do not. It is wrong often enough to quietly misattribute cost
   to the wrong department, which is worse than having no department at all.

Leaving it null is fine: GitHub pages still work, only the org breakdown is lost.

**`included_credits`** is a commercial term, not usage. It comes from your plan, and CreditLens
already holds it in the `GitHubBusinessStandardCredits` / `GitHubEnterpriseStandardCredits`
parameters. Set it here only if your agreement differs from the published allowance.

In [ ]:
# Route 1 - SAML name_id, where SSO is configured. Harmless if it is not.
try:
    consumed, page = [], 1
    while True:
        r = get(f"{API}/enterprises/{ENTERPRISE}/consumed-licenses",
                {"per_page": 100, "page": page})
        batch = r.json().get("users", [])
        if not batch:
            break
        consumed.extend(batch)
        page += 1

    upn = {u["github_com_login"]: u.get("saml_name_id")
           for u in consumed if u.get("saml_name_id")}
    print(f"{len(upn):,} of {len(seats):,} seats resolved to a SAML identity")

    if upn:
        mapping = spark.createDataFrame(
            [Row(username=k, upn_from_saml=v) for k, v in upn.items()])
        seat_df = (seat_df.join(mapping, "username", "left")
                          .withColumn("user_principal_name",
                                      F.coalesce("user_principal_name", "upn_from_saml"))
                          .drop("upn_from_saml"))
except Exception as e:
    print(f"SAML identity lookup unavailable ({e}).")
    print("Department breakdowns will be empty until user_principal_name is supplied.")

seat_df.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable(TBL_SEATS)
print(f"{TBL_SEATS}: {seat_df.count():,} seats")

## Usage — one call per user per month

The loop exists because `user` filters the request rather than appearing in the response.

In [ ]:
logins = [s["assignee"]["login"] for s in seats]
today = date.today()
months = [(today - relativedelta(months=i)) for i in range(BACKFILL_MONTHS)]

print(f"{len(logins):,} users x {len(months)} months = "
      f"{len(logins) * len(months):,} calls")

rows, empty = [], 0
for m in months:
    for i, login in enumerate(logins, 1):
        r = get(f"{API}/enterprises/{ENTERPRISE}/settings/billing/ai_credit/usage",
                {"user": login, "year": m.year, "month": m.month})
        items = r.json().get("usageItems", [])
        if not items:
            empty += 1
            continue
        for it in items:
            rows.append(Row(
                # The response carries no date beyond the period asked for, so
                # stamp the month. Day-grain needs a call per user PER DAY -
                # 500 users x 30 days is 15,000 calls a month, which exceeds
                # the hourly limit and is rarely worth it.
                usage_date=date(m.year, m.month, 1),
                username=login,
                product=it.get("product"),
                sku=it.get("sku"),
                model=it.get("model"),
                quantity=float(it.get("grossQuantity") or 0),
                unit_type=it.get("unitType"),
                applied_cost_per_quantity=float(it.get("pricePerUnit") or 0),
                gross_amount=float(it.get("grossAmount") or 0),
                discount_amount=float(it.get("discountAmount") or 0),
                net_amount=float(it.get("netAmount") or 0),
                organization=None,
                repository=None,          # not in the AI-credit schema
                cost_center_name=None,
                total_monthly_quota=None, # not available via API
            ))
        if i % 100 == 0:
            print(f"  {m:%Y-%m}: {i}/{len(logins)}")

print(f"\n{len(rows):,} usage rows; {empty:,} user-months with no usage")

In [ ]:
if not rows:
    print("No usage returned. Check the PAT scope and that the enterprise slug is right.")
else:
    usage = (spark.createDataFrame(rows)
             .withColumn("_loaded_at", F.current_timestamp()))

    # One row per user, month, sku and model. Re-running a month must update
    # rather than append, or a monthly backfill would double the spend.
    KEY = ["usage_date", "username", "sku", "model"]

    if spark.catalog.tableExists(TBL_USAGE):
        before = spark.table(TBL_USAGE).count()
        cond = " AND ".join(f"t.{k} <=> s.{k}" for k in KEY)
        (DeltaTable.forName(spark, TBL_USAGE).alias("t")
            .merge(usage.alias("s"), cond)
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute())
        after = spark.table(TBL_USAGE).count()
        print(f"{TBL_USAGE}: {before:,} -> {after:,}  (+{after - before:,})")
    else:
        usage.write.format("delta").saveAsTable(TBL_USAGE)
        print(f"{TBL_USAGE}: created with {usage.count():,} rows")

In [ ]:
spark.sql(f"""
    SELECT  MIN(usage_date)             AS earliest,
            MAX(usage_date)             AS latest,
            COUNT(DISTINCT username)    AS developers,
            COUNT(DISTINCT model)       AS models,
            ROUND(SUM(gross_amount), 2) AS gross,
            ROUND(SUM(net_amount), 2)   AS net_billable
    FROM    {TBL_USAGE}
""").show(truncate=False)

# net well below gross means the pooled allowance is absorbing most consumption,
# which is the normal and healthy state.
spark.sql(f"""
    SELECT  model,
            COUNT(DISTINCT username)    AS developers,
            ROUND(SUM(gross_amount), 2) AS gross,
            ROUND(SUM(net_amount), 2)   AS net
    FROM    {TBL_USAGE}
    GROUP BY model
    ORDER BY gross DESC
""").show(truncate=False)